In [1]:
### misc
import pandas as pd
import numpy as np
import os
from pathlib import Path
import pickle
import time
from itertools import product

#### graphical
import matplotlib.pyplot as plt
import corner

#### ML
import sklearn
from sklearn.decomposition import PCA
import tensorflow as tf
import keras
from keras import layers

from WMSE import WMSE, WMSE_metric

##### poke gpu
os.environ["CUDA_VISIBLE_DEVICES"]="0"

physical_devices = tf.config.list_physical_devices("GPU") 

tf.config.experimental.set_memory_growth(physical_devices[0], True)

gpu0usage = tf.config.experimental.get_memory_info("GPU:0")["current"]

print("Current GPU usage:\n"
     + " - GPU0: " + str(gpu0usage) + "B\n")

modelpath = '/home/hatte/M4/models'

2025-10-24 10:32:24.996510: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1761298345.008162 2476425 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1761298345.011668 2476425 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2025-10-24 10:32:25.024659: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


Current GPU usage:
 - GPU0: 0B



I0000 00:00:1761298346.374798 2476425 gpu_device.cc:2022] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 15482 MB memory:  -> device: 0, name: NVIDIA RTX A4500, pci bus id: 0000:41:00.0, compute capability: 8.6


In [2]:
def scheduler(epoch, lr,):
    ## Learning rate scheduler
    # Decreases learning rate in-training for stability
    if lr < 1e-5:
        return float(lr)
    else:
        return float(lr * tf.math.exp(-0.000006))

In [4]:
df_full = pd.read_hdf('../grids/Chiara-dnufit.hdf5', key='df') ## edit for your grid!!

df_full['logLPhot'] = np.log10(df_full['LPhot'])

df_full['lognumax'] = np.log10(df_full['numax']*3090)

df_full['logdnufit'] = np.log10(df_full['dnufit'])

df_full['logAge'] = np.log10(df_full['age'])

df_full['logTeff'] = np.log10(df_full['Teff'])

df_full['logmassini'] = np.log10(df_full['massini'])

df_full['logyini'] = np.log10(df_full['yini'])

df_full['logalphaMLT'] = np.log10(df_full['alphaMLT'])


#### define inputs
inputs = ['logmassini', 'zini', 'yini', 'alphaMLT', 'logAge', 'eta', 'alphaFe']

#### define outputs
classical_outputs = ['FeH', 'logLPhot', 'Teff']
astero_outputs = ['lognumax', 'logdnufit'] 

outputs = classical_outputs+astero_outputs

df = df_full[inputs+outputs]

df_norm = (df - df.min())/(df.max() - df.min())

## check df_norm.describe looks reasonable (min=0, max=1):
df_norm.describe()

#### train/test split with seed 
seed = 42

df_train = df_norm.sample(frac=0.95, random_state=seed)
df_test = df_norm.drop(df_train.index)

df_train_inputs, df_val_inputs, df_train_outputs, df_val_outputs = sklearn.model_selection.train_test_split(df_train[inputs],df_train[outputs], test_size = 0.05, random_state=seed)

print("Training set: ", len(df_train_inputs))
print("Validation set: ", len(df_val_inputs))
print("Test set: ", len(df_test))

Training set:  6754081
Validation set:  355478
Test set:  374187


In [6]:
#unnormed_weights_dict = {'FeH':0.01, 'logLPhot':0.001, 'Teff':10, 'numax':0.001/3090, 'dnuSer':0.0001/135}

unnormed_weights_dict = {'FeH':0.01, 'logLPhot':0.001, 'Teff':10, 'lognumax':0.001, 'logdnufit':0.0001}

unnormed_weights = list(unnormed_weights_dict.values())

weights = [2*unnormed_weights_dict[i]/(df[i].max() - df[i].min()) for i in outputs]

In [8]:
n_dense_layers = 6

dense_layer_units = 128

Nepochs = 400000

learning_rate = 0.0001

model_name = 'M4-log-mass-age-dnu-numax-Lphot-exponent-6e-6-Adam'

loss_func = 'WMSE'

df_train_inputs.join(df_train_outputs).to_hdf(f'{modelpath}/long-runs/training-data/training-{model_name}-nlayers-{n_dense_layers}-nunits-{dense_layer_units}-epochs-{Nepochs}-lrate-{learning_rate}-lossfunc-{loss_func}.hdf5', key = 'df')

In [9]:
checkpoint_dir = f'{modelpath}/long-runs/checkpoint/chk-{model_name}-nlayers-{n_dense_layers}-nunits-{dense_layer_units}-epochs-{Nepochs}-lrate-{learning_rate}-lossfunc-{loss_func}.model.keras'

full_model_dir = f'{modelpath}/long-runs/full-model/mod-{model_name}-nlayers-{n_dense_layers}-nunits-{dense_layer_units}-epochs-{Nepochs}-lrate-{learning_rate}-lossfunc-{loss_func}/'

if not os.path.exists(full_model_dir):
    os.makedirs(full_model_dir)

historyfile = f'{modelpath}/long-runs/history/hist-{model_name}-nlayers-{n_dense_layers}-nunits-{dense_layer_units}-epochs-{Nepochs}-lrate-{learning_rate}-lossfunc-{loss_func}.json'
    
cp_callback = tf.keras.callbacks.ModelCheckpoint(filepath = checkpoint_dir, verbose = 1, save_best_only = True, save_freq = 'epoch')

lr_callback = tf.keras.callbacks.LearningRateScheduler(scheduler, )

In [ ]:
######## map out model architecture
#### input layer
nn_input = keras.Input(shape=(len(inputs),))

#### dense layer(s)
for n_dense_layer in range(n_dense_layers):
    if n_dense_layer == 0:
        dense_layer = layers.Dense(dense_layer_units, activation='relu')(nn_input)
    else:
        dense_layer = layers.Dense(dense_layer_units, activation='relu')(dense_layer)

#### output layer
nn_output =  layers.Dense(len(outputs), activation='linear')(dense_layer)

######## store architecture as keras model
model = keras.Model(inputs=nn_input, outputs=nn_output, name=model_name)

tb_callback = tf.keras.callbacks.TensorBoard(log_dir = f'{modelpath}/logs/long-runs/log-{model_name}-nlayers-{n_dense_layers}-nunits-{dense_layer_units}-epochs-{Nepochs}-lrate-{learning_rate}-lossfunc-{loss_func}')

model.compile(loss=WMSE(weights), optimizer=tf.keras.optimizers.Adam(learning_rate=learning_rate))

history = model.fit(df_train_inputs,
          df_train_outputs,
          validation_data=(df_val_inputs,df_val_outputs),
          batch_size=2**14, #change higher
          verbose=1,
          epochs=Nepochs,
          shuffle=True, callbacks = [tb_callback, cp_callback, lr_callback]) 

tf.saved_model.save(model, full_model_dir)
hist_df = pd.DataFrame(history.history)
    
with open(os.path.join(modelpath, historyfile), mode="w") as f:
    hist_df.to_json(f)

Epoch 1/400000


I0000 00:00:1761298412.228609 2476653 service.cc:148] XLA service 0x75101c00f200 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1761298412.228645 2476653 service.cc:156]   StreamExecutor device (0): NVIDIA RTX A4500, Compute Capability 8.6
2025-10-24 10:33:32.275441: I tensorflow/compiler/mlir/tensorflow/utils/dump_mlir_util.cc:268] disabling MLIR crash reproducer, set env var `MLIR_CRASH_REPRODUCER_DIRECTORY` to enable.
I0000 00:00:1761298412.420419 2476653 cuda_dnn.cc:529] Loaded cuDNN version 90300
2025-10-24 10:33:32.496991: W external/local_xla/xla/service/gpu/nvptx_compiler.cc:930] The NVIDIA driver's CUDA version is 12.2 which is older than the PTX compiler version 12.5.82. Because the driver is older than the PTX compiler version, XLA is disabling parallel compilation, which may slow down compilation. You should update your NVIDIA driver or use the NVIDIA-provided CUDA forward compatibility packages.
2025-10-24 10:33:33.13115

 46/413 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - loss: 974831.9375 

I0000 00:00:1761298414.427229 2476653 device_compiler.h:188] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


398/413 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 650228.2500

2025-10-24 10:33:36.454512: I external/local_xla/xla/stream_executor/cuda/cuda_asm_compiler.cc:397] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_dot_258', 32 bytes spill stores, 32 bytes spill loads

2025-10-24 10:33:36.627494: I external/local_xla/xla/stream_executor/cuda/cuda_asm_compiler.cc:397] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_dot_446_0', 36 bytes spill stores, 36 bytes spill loads

2025-10-24 10:33:36.734156: I external/local_xla/xla/stream_executor/cuda/cuda_asm_compiler.cc:397] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_dot_446', 136 bytes spill stores, 136 bytes spill loads

2025-10-24 10:33:36.734589: I external/local_xla/xla/stream_executor/cuda/cuda_asm_compiler.cc:397] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_dot_446', 24 bytes spill stores, 24 bytes spill loads

2025-10-24 10:33:36.763179: I external/local_xla/xla/stream_

413/413 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - loss: 646687.7500

2025-10-24 10:33:39.042075: I external/local_xla/xla/stream_executor/cuda/cuda_asm_compiler.cc:397] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_dot_30', 32 bytes spill stores, 32 bytes spill loads

2025-10-24 10:33:39.134134: I external/local_xla/xla/stream_executor/cuda/cuda_asm_compiler.cc:397] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_dot_30_0', 168 bytes spill stores, 168 bytes spill loads




Epoch 1: val_loss improved from inf to 511229.15625, saving model to /home/hatte/M4/models/long-runs/checkpoint/chk-M4-log-mass-age-dnu-numax-Lphot-exponent-6e-6-Adam-nlayers-6-nunits-128-epochs-400000-lrate-0.0001-lossfunc-WMSE.model.keras
413/413 ━━━━━━━━━━━━━━━━━━━━ 8s 12ms/step - loss: 646459.2500 - val_loss: 511229.1562 - learning_rate: 9.9999e-05
Epoch 2/400000
402/413 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 510778.9688
Epoch 2: val_loss improved from 511229.15625 to 508499.65625, saving model to /home/hatte/M4/models/long-runs/checkpoint/chk-M4-log-mass-age-dnu-numax-Lphot-exponent-6e-6-Adam-nlayers-6-nunits-128-epochs-400000-lrate-0.0001-lossfunc-WMSE.model.keras
413/413 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step - loss: 510770.4375 - val_loss: 508499.6562 - learning_rate: 9.9999e-05
Epoch 3/400000
399/413 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 508021.7500
Epoch 3: val_loss improved from 508499.65625 to 503303.84375, saving model to /home/hatte/M4/models/long-runs/checkpoint/chk-M4-l

In [7]:
custom_objects =  {'WMSE':WMSE_metric}
model= tf.keras.models.load_model(checkpoint_dir, custom_objects = custom_objects)

In [8]:
n_learning_rate = model.optimizer.get_config()['learning_rate']

Nepochs = 200000

nepochs = 400000

tb_callback = tf.keras.callbacks.TensorBoard(log_dir = f'{modelpath}/logs/long-runs/log-{model_name}-nlayers-{n_dense_layers}-nunits-{dense_layer_units}-epochs-{Nepochs}-lrate-{learning_rate}-lossfunc-{loss_func}')

model.compile(loss=WMSE(weights), optimizer=tf.keras.optimizers.Adam(learning_rate=n_learning_rate))

In [ ]:
history = model.fit(df_train_inputs,
          df_train_outputs,
          validation_data=(df_val_inputs,df_val_outputs),
          batch_size=2**14, #change higher
          verbose=1,
          epochs=nepochs,
          shuffle=True, callbacks = [tb_callback, cp_callback, lr_callback],
          initial_epoch=Nepochs)



Epoch 200001/400000


I0000 00:00:1760950326.935602 3758390 service.cc:148] XLA service 0x770e44010080 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1760950326.935627 3758390 service.cc:156]   StreamExecutor device (0): NVIDIA RTX A4500, Compute Capability 8.6
2025-10-20 09:52:06.979298: I tensorflow/compiler/mlir/tensorflow/utils/dump_mlir_util.cc:268] disabling MLIR crash reproducer, set env var `MLIR_CRASH_REPRODUCER_DIRECTORY` to enable.
I0000 00:00:1760950327.120763 3758390 cuda_dnn.cc:529] Loaded cuDNN version 90300
2025-10-20 09:52:07.201048: W external/local_xla/xla/service/gpu/nvptx_compiler.cc:930] The NVIDIA driver's CUDA version is 12.2 which is older than the PTX compiler version 12.5.82. Because the driver is older than the PTX compiler version, XLA is disabling parallel compilation, which may slow down compilation. You should update your NVIDIA driver or use the NVIDIA-provided CUDA forward compatibility packages.
2025-10-20 09:52:07.78720

 49/413 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - loss: 17505.0918

I0000 00:00:1760950329.269984 3758390 device_compiler.h:188] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


401/413 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 4478.7925

2025-10-20 09:52:11.358120: I external/local_xla/xla/stream_executor/cuda/cuda_asm_compiler.cc:397] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_dot_258', 32 bytes spill stores, 32 bytes spill loads

2025-10-20 09:52:11.536111: I external/local_xla/xla/stream_executor/cuda/cuda_asm_compiler.cc:397] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_dot_446', 136 bytes spill stores, 136 bytes spill loads

2025-10-20 09:52:11.596941: I external/local_xla/xla/stream_executor/cuda/cuda_asm_compiler.cc:397] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_dot_446', 24 bytes spill stores, 24 bytes spill loads

2025-10-20 09:52:11.637375: I external/local_xla/xla/stream_executor/cuda/cuda_asm_compiler.cc:397] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_dot_370', 24 bytes spill stores, 48 bytes spill loads

2025-10-20 09:52:11.699798: I external/local_xla/xla/stream_ex

413/413 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - loss: 4383.1450

2025-10-20 09:52:14.032179: I external/local_xla/xla/stream_executor/cuda/cuda_asm_compiler.cc:397] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_dot_30', 32 bytes spill stores, 32 bytes spill loads

2025-10-20 09:52:14.096684: I external/local_xla/xla/stream_executor/cuda/cuda_asm_compiler.cc:397] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_dot_30_0', 168 bytes spill stores, 168 bytes spill loads




Epoch 200001: val_loss improved from inf to 152.22569, saving model to /home/hatte/M4/models/long-runs/checkpoint/chk-M4-logall-exponent-6e-6-Adam-nlayers-6-nunits-128-epochs-10000-lrate-0.0001-lossfunc-WMSE.model.keras
413/413 ━━━━━━━━━━━━━━━━━━━━ 9s 12ms/step - loss: 4375.3950 - val_loss: 152.2257 - learning_rate: 3.0377e-05
Epoch 200002/400000
404/413 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 145.9179
Epoch 200002: val_loss improved from 152.22569 to 147.51204, saving model to /home/hatte/M4/models/long-runs/checkpoint/chk-M4-logall-exponent-6e-6-Adam-nlayers-6-nunits-128-epochs-10000-lrate-0.0001-lossfunc-WMSE.model.keras
413/413 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step - loss: 145.9452 - val_loss: 147.5120 - learning_rate: 3.0376e-05
Epoch 200003/400000
411/413 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 182.6361
Epoch 200003: val_loss did not improve from 147.51204
413/413 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step - loss: 182.9377 - val_loss: 515.7785 - learning_rate: 3.0376e-05
Epoch 200004/400000


In [19]:
tf.saved_model.save(model, full_model_dir)
hist_df = pd.DataFrame(history.history)
    
with open(os.path.join(modelpath, historyfile), mode="w") as f:
    hist_df.to_json(f)

TypeError: this __dict__ descriptor does not support '_DictWrapper' objects

In [30]:
unnormed_weights_dict = {'FeH':0.001, 'logLPhot':0.001, 'Teff':0.1, 'numax':0.001/3090, 'dnuSer':0.001/135}

unnormed_weights = list(unnormed_weights_dict.values())

weights = [2*unnormed_weights_dict[i]/(df[i].max() - df[i].min()) for i in outputs]

In [31]:
n_learning_rate = model.optimizer.get_config()['learning_rate']

nnnepochs = 101000 + 50000

tb_callback = tf.keras.callbacks.TensorBoard(log_dir = f'{modelpath}/logs/long-runs/log-{model_name}-nlayers-{n_dense_layers}-nunits-{dense_layer_units}-epochs-{Nepochs}-lrate-{learning_rate}-lossfunc-{loss_func}')

model.compile(loss=WMSE(weights), optimizer=tf.keras.optimizers.Adam(learning_rate=n_learning_rate))

In [ ]:
history = model.fit(df_train_inputs,
          df_train_outputs,
          validation_data=(df_val_inputs,df_val_outputs),
          batch_size=2**14, #change higher
          verbose=1,
          epochs=nnnepochs,
          shuffle=True, callbacks = [tb_callback, cp_callback, lr_callback],
          initial_epoch=101000)

Epoch 101001/151000
413/413 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 1944704.0000
Epoch 101001: val_loss did not improve from 224604.18750
413/413 ━━━━━━━━━━━━━━━━━━━━ 4s 6ms/step - loss: 1941518.3750 - val_loss: 224674.1875 - learning_rate: 5.4440e-04
Epoch 101002/151000
398/413 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 221558.0000
Epoch 101002: val_loss did not improve from 224604.18750
413/413 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - loss: 221667.3281 - val_loss: 231856.9531 - learning_rate: 5.4440e-04
Epoch 101003/151000
405/413 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 228705.0156
Epoch 101003: val_loss did not improve from 224604.18750
413/413 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - loss: 228784.2344 - val_loss: 237591.3281 - learning_rate: 5.4440e-04
Epoch 101004/151000
412/413 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 246389.2812
Epoch 101004: val_loss did not improve from 224604.18750
413/413 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - loss: 246454.7656 - val_loss: 308530.3750 - learning_rate: 5.443

In [10]:
custom_objects =  {'WMSE':WMSE_metric}
tb_callback = tf.keras.callbacks.TensorBoard(log_dir = f'{modelpath}/logs/long-runs/log-{model_name}-nlayers-{n_dense_layers}-nunits-{dense_layer_units}-epochs-{Nepochs}-lrate-{learning_rate}-lossfunc-{loss_func}')

model = tf.keras.models.load_model(checkpoint_dir, custom_objects = custom_objects)
n_learning_rate = model.optimizer.get_config()['learning_rate']

In [14]:
model.compile(loss = WMSE(weights), optimizer=tf.keras.optimizers.Adam(learning_rate=0.001))

history = model.fit(df_train_inputs,
          df_train_outputs,
          validation_data=(df_val_inputs,df_val_outputs),
          batch_size=2**14, #change higher
          verbose=1,
          epochs=103624+1000,
          shuffle=True, callbacks = [tb_callback, cp_callback, lr_callback],
          initial_epoch=103624)

Epoch 103625/104624


2025-04-07 14:43:23.938591: I external/local_xla/xla/stream_executor/cuda/cuda_asm_compiler.cc:397] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_dot_451_0', 32 bytes spill stores, 32 bytes spill loads

2025-04-07 14:43:23.985739: I external/local_xla/xla/stream_executor/cuda/cuda_asm_compiler.cc:397] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_dot_461', 20 bytes spill stores, 20 bytes spill loads

2025-04-07 14:43:24.037567: I external/local_xla/xla/stream_executor/cuda/cuda_asm_compiler.cc:397] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_dot_451', 192 bytes spill stores, 192 bytes spill loads

2025-04-07 14:43:24.165488: I external/local_xla/xla/stream_executor/cuda/cuda_asm_compiler.cc:397] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_dot_461', 416 bytes spill stores, 420 bytes spill loads

2025-04-07 14:43:24.180500: I external/local_xla/xla/strea

407/413 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: nan

2025-04-07 14:43:26.684664: I external/local_xla/xla/stream_executor/cuda/cuda_asm_compiler.cc:397] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_dot_451', 24 bytes spill stores, 24 bytes spill loads

2025-04-07 14:43:26.729924: I external/local_xla/xla/stream_executor/cuda/cuda_asm_compiler.cc:397] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_dot_451', 136 bytes spill stores, 136 bytes spill loads

2025-04-07 14:43:26.770750: I external/local_xla/xla/stream_executor/cuda/cuda_asm_compiler.cc:397] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_dot_461', 24 bytes spill stores, 48 bytes spill loads

2025-04-07 14:43:26.835583: I external/local_xla/xla/stream_executor/cuda/cuda_asm_compiler.cc:397] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_dot_461', 412 bytes spill stores, 416 bytes spill loads

2025-04-07 14:43:26.894882: I external/local_xla/xla/stream_

413/413 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - loss: nan
Epoch 103625: val_loss did not improve from inf
413/413 ━━━━━━━━━━━━━━━━━━━━ 6s 9ms/step - loss: nan - val_loss: nan - learning_rate: 9.9999e-04
Epoch 103626/104624
411/413 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: nan
Epoch 103626: val_loss did not improve from inf
413/413 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - loss: nan - val_loss: nan - learning_rate: 9.9999e-04
Epoch 103627/104624
255/413 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: nan

KeyboardInterrupt: 